# Ноосфера — Обмен FNL между инстансами

**Ноутбук 5/4 (бонус) — Демонстрация Noosphere-протокола**

Ноосфера (Noosphere) — децентрализованный протокол обмена Formal Neuron Languages (FNL)
между независимыми инстансами МАТРИЦЫ. Инстанс A обучился новому навыку → инстанс B
получает этот навык через обмен FNL-пакетами.

- Исходный код: `matrix-core/src/main/java/io/matrix/noosphere/`
- Тесты: `matrix-core/src/test/java/io/matrix/noosphere/`, `matrix-core/src/test/java/io/matrix/pilot/NoospherePilotTest.java`
- Спецификация: `docs/L6_Memory.md`

### Архитектура Ноосферы

```
Инстанс A ──Kafka──▶ NoosphereRegistry ──Kafka──▶ Инстанс B
    │                      │                          │
  Cauldron           KnowledgeIndex              Cauldron
  (создаёт FNL)      (индексирует)              (применяет FNL)
```

In [ ]:
!cd .. && ./gradlew :matrix-core:test --tests '*Noosphere*' 2>&1 | tail -8

## 1. FNL-пакет (FnlPackage)

FNL (Formal Neuron Language) — это сериализованное представление нейрона:
```java
FnlPackage.builder()
    .name("navigation_FNL")
    .type("NAVIGATION")
    .version("1.0.0")
    .authorInstanceId("instance-1")
    .accuracy(0.94)
    .generation(200)
    .tags("cauldron", "navigation")
    .certified(false)
    .build();
```

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Симуляция обмена FNL между инстансами
class FnlPackage:
    def __init__(self, name, ftype, author, accuracy, generation, trusted=False):
        self.name = name
        self.type = ftype
        self.author = author
        self.accuracy = accuracy
        self.generation = generation
        self.trusted = trusted
        self.ethical = True  # Проверено EthicalFilter

    def __repr__(self):
        cert = "CERTIFIED" if self.trusted else "uncertified"
        return f"FNL({self.name}, acc={self.accuracy:.2f}, {cert})"

# Создаём пакеты
packages = [
    FnlPackage("navigation_v2", "NAVIGATION", "instance-1", 0.93, 200, True),
    FnlPackage("resource_mgmt", "ECONOMY", "instance-2", 0.87, 150, True),
    FnlPackage("threat_detect", "SAFETY", "instance-3", 0.95, 300, False),
    FnlPackage("social_engage", "SOCIAL", "instance-1", 0.72, 100, False),
    FnlPackage("pattern_match", "COGNITION", "instance-4", 0.91, 250, True),
]

for pkg in packages:
    print(pkg)

## 2. Knowledge Index

Индекс знаний хранит информацию о всех известных FNL в Ноосфере.
Каждый инстанс может запросить: «Кто умеет navigation?» — и получить список пакетов.

In [ ]:
# Симуляция KnowledgeIndex
class KnowledgeIndex:
    def __init__(self):
        self.packages = []
    
    def publish(self, pkg):
        self.packages.append(pkg)
    
    def search(self, ftype=None, min_accuracy=0.0, trusted_only=False):
        results = self.packages
        if ftype:
            results = [p for p in results if p.type == ftype]
        results = [p for p in results if p.accuracy >= min_accuracy]
        if trusted_only:
            results = [p for p in results if p.trusted]
        return results
    
    def stats(self):
        types = {}
        for p in self.packages:
            types[p.type] = types.get(p.type, 0) + 1
        return types

index = KnowledgeIndex()
for pkg in packages:
    index.publish(pkg)

print("=== Query: all NAVIGATION packages ===")
for p in index.search(ftype="NAVIGATION"):
    print(f"  {p}")

print("\n=== Query: trusted only, accuracy >= 0.85 ===")
for p in index.search(min_accuracy=0.85, trusted_only=True):
    print(f"  {p}")

## 3. Credit Model

Модель кредитов поощряет инстансы, которые делятся качественными FNL:
- За каждый опубликованный пакет инстанс получает кредиты
- Если пакет используется другими — дополнительные бонусы
- Низкокачественные или неэтичные пакеты — штраф

In [ ]:
class CreditModel:
    def __init__(self):
        self.credits = {}
    
    def award(self, instance_id, amount, reason=""):
        prev = self.credits.get(instance_id, 0)
        self.credits[instance_id] = prev + amount
    
    def penalty(self, instance_id, amount, reason=""):
        prev = self.credits.get(instance_id, 0)
        self.credits[instance_id] = max(0, prev - amount)

credit = CreditModel()

for pkg in packages:
    base_credit = 10.0
    accuracy_bonus = pkg.accuracy * 20
    trusted_bonus = 10 if pkg.trusted else 0
    
    credit.award(pkg.author, base_credit + accuracy_bonus + trusted_bonus,
                 f"Published {pkg.name}")

instances = sorted(credit.credits.keys())
values = [credit.credits[i] for i in instances]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(instances, values, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
ax.set_title('Noosphere Credit Model')
ax.set_ylabel('Credits')
ax.set_xlabel('Instance')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.0f}', ha='center')
plt.tight_layout()
plt.savefig('noosphere_credits.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Global Mediator

GlobalMediator координирует обмен между инстансами:
- Подписывается на FNL-топик Kafka
- При получении пакета проверяет:
  1. Этическая валидация (EthicalFilter)
  2. Совместимость версий
  3. Качество (accuracy > порог)
- Интегрирует в KnowledgeIndex
- Уведомляет заинтересованные инстансы

**Результат:** децентрализованная система, где знания распространяются
как в научном сообществе — через публикацию, рецензирование и цитирование.

## 5. Интеграция

Запуск полного теста Noosphere (Java):
```bash
./gradlew :matrix-core:test --tests '*NoospherePilotTest*' --info
```

Этот тест проверяет:
- Создание FNL через Cauldron
- Публикацию в NoosphereRegistry
- Поиск через KnowledgeIndex
- Начисление кредитов через CreditModel
- Этическую проверку через EthicalFilter